#### Build Machine Learning Model

In [ ]:
import ast
import os

import neurokit2 as nk
import numpy as np
import pandas as pd
import wfdb

In [ ]:
# create mock dataset

# connect dataset
# PTB-XL folder in the project root (git-ignored; a symlink to the dataset works)
ecg_db_path = os.path.join("..", "ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.0.3")
sampling_rate = 100  # 100Hz -> filename_lr (records100/), 500Hz -> filename_hr (records500/)

# load metadata; scp_codes is stored as a dict string
Y = pd.read_csv(os.path.join(ecg_db_path, "ptbxl_database.csv"), index_col="ecg_id")
Y.scp_codes = Y.scp_codes.apply(ast.literal_eval)

# recording_date is stored as text -> parse to datetime (the only timestamp; .hea headers have none)
Y.recording_date = pd.to_datetime(Y.recording_date, format="%Y-%m-%d %H:%M:%S")

# ages over 89 are stored as 300 for privacy -> treat as missing
Y.loc[Y.age == 300, "age"] = np.nan

# build the full path to each 100Hz record (wfdb expects the path without the .dat/.hea extension)
Y["path_100hz"] = Y.filename_lr.apply(lambda f: os.path.join(ecg_db_path, f))
patient_cols = ["patient_id", "recording_date", "age", "sex", "height", "weight", "pacemaker",
                "scp_codes", "heart_axis", "infarction_stadium1", "strat_fold", "filename_lr"]
Y[patient_cols].head()

In [ ]:
# map scp_codes to diagnostic superclasses (NORM, MI, STTC, CD, HYP)
agg_df = pd.read_csv(os.path.join(ecg_db_path, "scp_statements.csv"), index_col=0)
agg_df = agg_df[agg_df.diagnostic == 1]  # keep only diagnostic statements (drops form/rhythm codes)


def aggregate_diagnostic(scp_codes):
    return sorted({agg_df.loc[code, "diagnostic_class"] for code in scp_codes if code in agg_df.index})


Y["diagnostic_superclass"] = Y.scp_codes.apply(aggregate_diagnostic)
Y[["scp_codes", "diagnostic_superclass"]].head()

In [ ]:
# check if the same patient has multiple ECGs
ecgs_per_patient = Y.groupby("patient_id").size()
print(f"{len(Y)} ECGs from {ecgs_per_patient.size} patients; "
      f"{(ecgs_per_patient > 1).sum()} patients have more than one ECG")

# strat_fold keeps all ECGs of a patient in one fold -> no patient leakage between train/val/test
assert Y.groupby("patient_id").strat_fold.nunique().max() == 1
ecgs_per_patient.value_counts().sort_index().rename_axis("ECGs per patient").to_frame("patients")

In [ ]:
# order each patient's recordings by date/time
Y["recording_day"] = Y.recording_date.dt.date
Y["recording_time"] = Y.recording_date.dt.time
Y = Y.sort_values(["patient_id", "recording_date"])
Y["visit_number"] = Y.groupby("patient_id").cumcount() + 1                      # 1 = first ECG of that patient
Y["time_since_prev"] = Y.groupby("patient_id").recording_date.diff()           # NaT for the first ECG
Y["days_since_first"] = (Y.recording_date - Y.groupby("patient_id").recording_date.transform("min")).dt.days
Y = Y.sort_index()  # back to ecg_id order so rows line up with X

# example: all recordings of the patient with the most ECGs, in time order
pid = ecgs_per_patient.idxmax()
Y[Y.patient_id == pid].sort_values("recording_date")[
    ["patient_id", "recording_date", "visit_number", "time_since_prev", "days_since_first", "diagnostic_superclass"]]

In [ ]:
# for patients with exactly two ECGs: time difference between the two recordings
two_ecg = Y[Y.patient_id.isin(ecgs_per_patient[ecgs_per_patient == 2].index)]
gap = two_ecg[two_ecg.visit_number == 2].set_index("patient_id").time_since_prev
print(f"{len(gap)} patients with two ECGs")
print(gap.describe())

gap_days = gap.dt.total_seconds() / 86400
bins = [-1e-9, 1 / 24, 1, 7, 30, 90, 365, np.inf]
labels = ["< 1 hour", "1 hour - 1 day", "1 - 7 days", "1 - 4 weeks", "1 - 3 months", "3 - 12 months", "> 1 year"]
pd.cut(gap_days, bins, labels=labels).value_counts().sort_index().rename_axis("time between ECGs").to_frame("patients")

In [ ]:
def load_raw_data(paths):
    """Read each record's signal -> array of shape (n_records, 1000, 12)."""
    return np.array([wfdb.rdsamp(p)[0] for p in paths])


X = load_raw_data(Y.path_100hz)
X.shape

In [ ]:
# check signal length: every record should be 10 s at 100Hz -> 1000 samples x 12 leads
n_records, sig_len, n_leads = X.shape
print(f"{n_records} records, {sig_len} samples ({sig_len / sampling_rate:.0f} s at {sampling_rate}Hz), {n_leads} leads")
assert (sig_len, n_leads) == (10 * sampling_rate, 12)

In [ ]:
# HRV for patients with two ECGs (10 s recordings -> ultra-short-term time-domain HRV only)
LEAD_II = 1  # lead order: I, II, III, AVR, AVL, AVF, V1-V6


def compute_hrv(ecg, fs=sampling_rate):
    """R-peaks from NeuroKit2 -> RR intervals (ms) -> time-domain HRV."""
    try:
        clean = nk.ecg_clean(ecg, sampling_rate=fs)
        _, info = nk.ecg_peaks(clean, sampling_rate=fs, correct_artifacts=True)
        rr = np.diff(info["ECG_R_Peaks"]) / fs * 1000
    except Exception:
        rr = np.array([])
    if len(rr) < 5:  # too few beats for a meaningful HRV value
        return {"n_beats": len(rr) + 1 if len(rr) else 0}
    return {
        "n_beats": len(rr) + 1,
        "mean_rr_ms": rr.mean(),
        "mean_hr_bpm": 60000 / rr.mean(),
        "sdnn_ms": rr.std(ddof=1),
        "rmssd_ms": np.sqrt(np.mean(np.diff(rr) ** 2)),
        "pnn50_pct": np.mean(np.abs(np.diff(rr)) > 50) * 100,
    }


hrv_ids = two_ecg.sort_values(["patient_id", "recording_date"]).index
hrv_rows = X[Y.index.get_indexer(hrv_ids), :, LEAD_II]  # X rows are in Y (ecg_id) order
hrv = pd.DataFrame([compute_hrv(ecg) for ecg in hrv_rows], index=hrv_ids)

# HRV is only meaningful in sinus rhythm; AFIB / extra beats inflate SDNN and RMSSD
Y["sinus_rhythm"] = Y.scp_codes.apply(lambda codes: "SR" in codes)
hrv_df = Y.loc[hrv_ids, ["patient_id", "visit_number", "recording_date", "time_since_prev",
                         "age", "sex", "diagnostic_superclass", "sinus_rhythm"]].join(hrv)
hrv_df.to_csv("hrv_two_ecg_patients.csv")
print(f"{len(hrv_df)} ECGs, {hrv_df.sdnn_ms.isna().sum()} without enough beats for HRV, "
      f"{hrv_df.sinus_rhythm.sum()} in sinus rhythm")
hrv_df.head(10)

In [ ]:
# compare HRV between the 1st and 2nd ECG of each patient (pNN50 left out)
from scipy.stats import wilcoxon

hrv_metrics = ["mean_rr_ms", "mean_hr_bpm", "sdnn_ms", "rmssd_ms"]
wide = hrv_df.pivot(index="patient_id", columns="visit_number", values=hrv_metrics + ["sinus_rhythm"])
wide.columns = [f"{m}_{v}" for m, v in wide.columns]

hrv_compare = pd.DataFrame({
    "days_between": hrv_df[hrv_df.visit_number == 2].set_index("patient_id").time_since_prev.dt.days,
    "both_sinus": wide.sinus_rhythm_1.astype(bool) & wide.sinus_rhythm_2.astype(bool),
})
for m in hrv_metrics:
    first, second = wide[f"{m}_1"].astype(float), wide[f"{m}_2"].astype(float)
    hrv_compare[f"{m}_1"] = first
    hrv_compare[f"{m}_2"] = second
    hrv_compare[f"{m}_change"] = second - first  # 2nd ECG minus 1st
hrv_compare.to_csv("hrv_compare_two_ecg.csv")

# paired summary, only patients in sinus rhythm on both ECGs with HRV on both
pairs = hrv_compare[hrv_compare.both_sinus].dropna(subset=[f"{m}_change" for m in hrv_metrics])
summary = pd.DataFrame({
    m: {
        "median_1st": pairs[f"{m}_1"].median(),
        "median_2nd": pairs[f"{m}_2"].median(),
        "median_change": pairs[f"{m}_change"].median(),
        "median_abs_change": pairs[f"{m}_change"].abs().median(),
        "wilcoxon_p": wilcoxon(pairs[f"{m}_1"], pairs[f"{m}_2"]).pvalue,
    }
    for m in hrv_metrics
}).T.round(3)
print(f"{len(hrv_compare)} patients, {len(pairs)} in sinus rhythm on both ECGs")
summary

In [ ]:
# similarity between each patient's two ECGs: raw signal vs median beat vs HRV
# test: is a patient's own pair more similar than a mismatched pair (1st ECG vs another patient's 2nd)?
from scipy.signal import butter, filtfilt
from scipy.spatial.distance import mahalanobis
from sklearn.metrics import roc_auc_score

b_bp, a_bp = butter(2, [0.5, 40], btype="band", fs=sampling_rate)  # remove baseline drift + noise


def filtered(ecg_id):
    return filtfilt(b_bp, a_bp, X[Y.index.get_loc(ecg_id)], axis=0)  # (1000, 12)


def median_beat(sig, before=0.25, after=0.45):
    """Median heartbeat per lead, aligned on lead II R-peaks -> (70, 12) at 100Hz."""
    pre, post = int(before * sampling_rate), int(after * sampling_rate)
    try:
        _, info = nk.ecg_peaks(sig[:, LEAD_II], sampling_rate=sampling_rate, correct_artifacts=True)
    except Exception:
        return None
    peaks = [r for r in info["ECG_R_Peaks"] if r - pre >= 0 and r + post <= len(sig)]
    if len(peaks) < 2:
        return None
    return np.median([sig[r - pre:r + post] for r in peaks], axis=0)


def signal_measures(a, b):
    a, b = a.ravel(), b.ravel()
    return {
        "euclidean": np.linalg.norm(a - b),
        "manhattan": np.abs(a - b).sum(),
        "cosine": a @ b / (np.linalg.norm(a) * np.linalg.norm(b)),
        "pearson": np.corrcoef(a, b)[0, 1],
    }


# HRV vectors: z-score each metric so mean_rr (ms) doesn't dominate; Mahalanobis also handles RR <-> HR correlation
hrv_z = hrv_df[hrv_metrics].dropna()
hrv_z = (hrv_z - hrv_z.mean()) / hrv_z.std()
inv_cov = np.linalg.pinv(np.cov(hrv_z.T))


def hrv_measures(a, b):
    return {
        "euclidean": np.linalg.norm(a - b),
        "manhattan": np.abs(a - b).sum(),
        "cosine": a @ b / (np.linalg.norm(a) * np.linalg.norm(b)),
        "mahalanobis": mahalanobis(a, b, inv_cov),
    }


# pairs: own pair (patient i: ECG1 vs ECG2) and mismatched pair (patient i ECG1 vs patient j ECG2, j != i)
visits = hrv_df.reset_index().pivot(index="patient_id", columns="visit_number", values="ecg_id")
rng = np.random.default_rng(0)
order = rng.permutation(len(visits))
partner = np.empty(len(visits), dtype=int)
partner[order] = np.roll(order, -1)  # every patient gets a different patient's 2nd ECG
pairs_df = pd.concat([
    pd.DataFrame({"patient_id": visits.index, "ecg_1": visits[1].values, "ecg_2": visits[2].values, "pair": "same patient"}),
    pd.DataFrame({"patient_id": visits.index, "ecg_1": visits[1].values, "ecg_2": visits[2].values[partner], "pair": "different patients"}),
], ignore_index=True)

sig_cache = {eid: filtered(eid) for eid in hrv_df.index}
beat_cache = {eid: median_beat(sig) for eid, sig in sig_cache.items()}

rows = []
for r in pairs_df.itertuples():
    row = {f"raw signal | {k}": v for k, v in signal_measures(sig_cache[r.ecg_1], sig_cache[r.ecg_2]).items()}
    b1, b2 = beat_cache[r.ecg_1], beat_cache[r.ecg_2]
    if b1 is not None and b2 is not None:
        row |= {f"median beat | {k}": v for k, v in signal_measures(b1, b2).items()}
    if r.ecg_1 in hrv_z.index and r.ecg_2 in hrv_z.index:
        row |= {f"HRV | {k}": v for k, v in hrv_measures(hrv_z.loc[r.ecg_1].values, hrv_z.loc[r.ecg_2].values).items()}
    rows.append(row)
similarity_df = pairs_df.join(pd.DataFrame(rows))
similarity_df.to_csv("ecg_pair_similarity.csv", index=False)

# AUC: chance that an own pair scores more similar than a mismatched pair (0.5 = no better than chance)
measure_cols = [c for c in similarity_df.columns if " | " in c]
higher_is_similar = {"cosine", "pearson"}
auc_rows = []
for col in measure_cols:
    d = similarity_df[["pair", col]].dropna()
    score = d[col] if col.split(" | ")[1] in higher_is_similar else -d[col]
    auc_rows.append({"measure": col, "auc": roc_auc_score(d.pair == "same patient", score), "n_pairs": len(d)})
auc_df = pd.DataFrame(auc_rows).sort_values("auc", ascending=False).reset_index(drop=True)
auc_df

In [ ]:
# plot: distribution of each measure for own vs mismatched pairs, then AUC ranking
import matplotlib.pyplot as plt

INK, INK_2, MUTED, GRID, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#fcfcfb"
PAIR_COLORS = {"same patient": "#2a78d6", "different patients": "#eb6834"}
plt.rcParams.update({
    "figure.facecolor": SURFACE, "axes.facecolor": SURFACE, "axes.edgecolor": "#c3c2b7",
    "axes.labelcolor": INK_2, "xtick.color": MUTED, "ytick.color": MUTED, "text.color": INK,
    "axes.spines.top": False, "axes.spines.right": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": GRID, "grid.linewidth": 0.6, "font.size": 9, "axes.titlesize": 10, "axes.axisbelow": True,
})

auc_by_measure = auc_df.set_index("measure").auc
groups = ["raw signal", "median beat", "HRV"]
fig, axes = plt.subplots(len(groups), 4, figsize=(15, 9.5), constrained_layout=True)
for row_axes, group in zip(axes, groups):
    cols = [c for c in measure_cols if c.startswith(group)]
    for ax, col in zip(row_axes, cols):
        d = similarity_df[["pair", col]].dropna()
        lo, hi = d[col].quantile([0.01, 0.99])  # clip long tails so the bulk is readable
        bins = np.linspace(lo, hi, 40)
        for pair, color in PAIR_COLORS.items():
            ax.hist(d.loc[d.pair == pair, col].clip(lo, hi), bins=bins, density=True,
                    histtype="step", linewidth=2, color=color, label=pair)
        better = "higher" if col.split(" | ")[1] in higher_is_similar else "lower"
        ax.set_title(f"{col}\nAUC {auc_by_measure[col]:.2f}  ({better} = more similar)", loc="left")
        ax.set_yticks([])
fig.legend(*axes[0, 0].get_legend_handles_labels(), frameon=False, loc="outside upper right", ncols=2)
fig.suptitle("Own ECG pair vs mismatched pair: distribution of each similarity measure", x=0.01, ha="left", fontsize=12)
fig.savefig("ecg_pair_similarity_distributions.png", dpi=120)
plt.show()

fig, ax = plt.subplots(figsize=(8, 5), constrained_layout=True)
ranked = auc_df.iloc[::-1]
ax.barh(ranked.measure, ranked.auc - 0.5, left=0.5, height=0.6, color="#2a78d6")
for y, v in enumerate(ranked.auc):
    ax.text(max(v, 0.5) + 0.005, y, f"{v:.2f}", va="center", color=INK_2)  # outside the bar, also when AUC < 0.5
ax.axvline(0.5, color=MUTED, linewidth=1, linestyle="--")
ax.text(0.5, -0.9, "chance (0.5)", color=MUTED, ha="center")
ax.set_xlim(0.4, 1)
ax.set_ylim(-1.2, len(ranked) - 0.5)
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
ax.set_xlabel("AUC: own pair more similar than mismatched pair")
ax.set_title("Which similarity measure recognises the same patient?", loc="left")
fig.savefig("ecg_pair_similarity_auc.png", dpi=120)
plt.show()

#### Data cleaning test

The first clustering (lead-II beat detection, no quality filter) produced a group whose average beat peaked ~20 ms after the alignment point. This section tests each cleaning step on every patient's first ECG (plus both ECGs of the 2-ECG patients) before clustering again.

**Process**
1. **Filter**: 0.5–40 Hz band-pass on all 12 leads (removes baseline drift and high-frequency noise).
2. **Beat detection**: compare two methods
   - *lead II* (old): R-peaks found on lead II only. Fails when the QRS is mostly negative in lead II.
   - *all 12 leads* (new): R-peaks found on the root-sum-square of all leads, then snapped to the QRS energy peak (±50 ms). Doesn't depend on which way the QRS points in a single lead.
3. **Median beat**: −250 ms to +450 ms around each beat, median across beats → 70 samples × 12 leads.
4. **Quality checks per ECG**
   - *alignment error*: where the QRS energy peak lands in the median beat (0 ms = correctly aligned)
   - *beat consistency*: median Pearson r between each beat and the median beat
   - *heart rate*: from the median RR interval (should be 30–200 bpm)
   - *amplitude* and *flat leads*, plus the dataset's `pacemaker` / `electrodes_problems` flags
5. **Exclusion rules**: no beat found, pacemaker, electrode problem, flat lead, amplitude > 10 mV, beat consistency < 0.8.
   The dataset's noise flags (`static_noise`, `burst_noise`, `baseline_drift`) are **not** used: their beats are as consistent as unflagged ECGs (median r 0.976 vs 0.978), so excluding them would discard good data.
6. **Validation**: re-run the same-patient test (own pair vs mismatched pair, ROC AUC) for every combination of beat detection × quality filter × similarity (plain Pearson vs shift-tolerant Pearson, the best r over shifts of ±50 ms).

In [ ]:
# test the cleaning process: lead-II beat detection (used above) vs all-lead detection + quality checks
from watch_pipeline import SNAP, beat_template  # the cleaning process lives in watch_pipeline.py (shared with the API)

qc_ids = Y.index[Y.visit_number == 1].union(hrv_df.index)  # every patient's 1st ECG + both ECGs of 2-ECG patients
templates = {"lead_ii": {}, "rss": {}}
qc_rows = []
for eid in qc_ids:
    sig = filtered(eid)
    row = {"ecg_id": eid, "max_abs_mv": np.abs(sig).max(), "flat_leads": int((sig.std(axis=0) < 0.01).sum())}
    for mode in templates:
        t = beat_template(sig, detect_on=mode)
        templates[mode][eid] = None if t is None else t["beat"]
        for k in ["consistency", "qrs_offset_ms", "heart_rate"]:
            row[f"{mode}_{k}"] = np.nan if t is None else t[k]
    qc_rows.append(row)
qc = pd.DataFrame(qc_rows).set_index("ecg_id")

# exclusion rules (each tested on the data; noise flags are NOT used: their beats are as consistent as the rest)
exclusions = pd.DataFrame({
    "no beat found": qc.rss_consistency.isna(),
    "pacemaker": Y.loc[qc.index, "pacemaker"].notna(),
    "electrode problem": Y.loc[qc.index, "electrodes_problems"].notna(),
    "flat lead": qc.flat_leads > 0,
    "amplitude > 10 mV": qc.max_abs_mv > 10,
    "inconsistent beats (r < 0.8)": qc.rss_consistency < 0.8,
})
qc["usable"] = ~exclusions.any(axis=1)

first_qc = qc.loc[Y.index[Y.visit_number == 1]]
print("first ECG per patient:", len(first_qc), "| excluded:", (~first_qc.usable).sum())
print(exclusions.loc[first_qc.index].sum().to_string())
pd.DataFrame({
    mode: {
        "QRS misaligned >= 30 ms": (first_qc[f"{mode}_qrs_offset_ms"].abs() >= 30).sum(),
        "heart rate outside 30-200": (~first_qc[f"{mode}_heart_rate"].between(30, 200)).sum(),
        "median beat consistency": first_qc[f"{mode}_consistency"].median().round(3),
    } for mode in templates
})

In [ ]:
# does the cleaning help? re-run the same-patient test (own pair vs mismatched pair, same pairs as above)


from watch_pipeline import pearson, shift_pearson

variant_rows = []
for mode in templates:
    for filtered_only in [False, True]:
        for sim_name, sim in [("pearson", pearson), ("shift-tolerant pearson", shift_pearson)]:
            scores, is_same = [], []
            for r in pairs_df.itertuples():
                t1, t2 = templates[mode][r.ecg_1], templates[mode][r.ecg_2]
                if t1 is None or t2 is None or (filtered_only and not (qc.usable[r.ecg_1] and qc.usable[r.ecg_2])):
                    continue
                scores.append(sim(t1, t2)); is_same.append(r.pair == "same patient")
            variant_rows.append({"beat detection": mode, "quality filter": filtered_only, "similarity": sim_name,
                                 "pairs": len(scores), "auc": roc_auc_score(is_same, scores)})
clean_variants = pd.DataFrame(variant_rows).sort_values("auc", ascending=False).reset_index(drop=True)
clean_variants.round(4)

In [ ]:
# plot the cleaning test: alignment error per detection method, and same-patient AUC per cleaning variant
fig, (ax_off, ax_auc) = plt.subplots(1, 2, figsize=(15, 4.8), width_ratios=[1, 1.4], constrained_layout=True)
offsets = np.arange(-60, 70, 10)
width = 4
for j, (mode, color, label) in enumerate([("lead_ii", "#eb6834", "lead II (old)"), ("rss", "#2a78d6", "all 12 leads (new)")]):
    counts = first_qc[f"{mode}_qrs_offset_ms"].round(-1).value_counts().reindex(offsets, fill_value=0)
    ax_off.bar(offsets + (j - 0.5) * width, counts, width=width * 0.9, color=color, label=label)
ax_off.set_yscale("log")
ax_off.set_xlabel("QRS peak position in median beat (ms from alignment point)")
ax_off.set_ylabel("patients (log scale)")
ax_off.set_title("Beat alignment error", loc="left")
ax_off.legend(frameon=False)

v = clean_variants.iloc[::-1]
names = [f"{d.replace('_', ' ')} | {'filtered' if f else 'all ECGs'} | {s}" for d, f, s in
         zip(v["beat detection"], v["quality filter"], v["similarity"])]
ax_auc.barh(names, v.auc - 0.9, left=0.9, height=0.6, color="#2a78d6")
for y, a in enumerate(v.auc):
    ax_auc.text(a + 0.001, y, f"{a:.3f}", va="center", color=INK_2)
ax_auc.set_xlim(0.9, 0.98)
ax_auc.grid(axis="y", visible=False); ax_auc.grid(axis="x", visible=True)
ax_auc.set_xlabel("AUC: own pair more similar than mismatched pair")
ax_auc.set_title("Same-patient test per cleaning variant (median beat)", loc="left")
fig.savefig("cleaning_test.png", dpi=120)
plt.show()

#### Data cleaning test: results

**Beat detection** (first ECG per patient, n = 18,869)

| Check | Lead II (old) | All 12 leads (new) |
|---|---|---|
| QRS misaligned by ≥ 30 ms | **1,193** | **67** |
| Heart rate outside 30–200 bpm | 12 | 0 |
| Median beat consistency | 0.977 | 0.978 |

**Excluded by the quality filter**: 642 of 18,869 (3.4%)

| Reason | ECGs |
|---|---|
| Inconsistent beats (r < 0.8) | 403 |
| Pacemaker | 218 |
| Electrode problem | 25 |
| Amplitude > 10 mV | 13 |
| Flat lead | 1 |

(An ECG can meet more than one rule.)

**Same-patient test** (AUC: own pair more similar than mismatched pair)

| Beat detection | Quality filter | Similarity | AUC |
|---|---|---|---|
| all 12 leads | yes | shift-tolerant Pearson | **0.970** |
| lead II | yes | shift-tolerant Pearson | 0.957 |
| all 12 leads | no | shift-tolerant Pearson | 0.954 |
| lead II | yes | Pearson | 0.949 |
| lead II | no | shift-tolerant Pearson | 0.944 |
| all 12 leads | yes | Pearson | 0.941 |
| lead II | no | Pearson (first attempt) | 0.937 |
| all 12 leads | no | Pearson | 0.927 |

**Takeaways**
- Lead-II detection was the main problem: it misaligned 1,193 patients' beats; all-lead detection leaves 67.
- The best cleaning (all-lead detection + quality filter + shift-tolerant similarity) raises the AUC from **0.937 to 0.970**.
- All-lead detection *with plain Pearson* scores lower than lead II. Lead-II misalignment is repeated in both of a patient's ECGs, so it acts like a personal fingerprint and inflates the same-patient score; it still distorts comparisons between different patients. Only shift-tolerant similarity separates real beat shape from alignment.
- So the clustering below uses all-lead detection, the quality filter, and every beat shifted (±50 ms) onto a common reference beat.

In [ ]:
# group all patients by median-beat shape again, with the cleaned process:
# all-lead beat detection, quality filter, and every beat shifted (+-50 ms) onto a common reference
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

beat_ids = first_qc.index[first_qc.usable].values
raw_beats = np.stack([templates["rss"][eid] for eid in beat_ids])  # (n_patients, 70, 12)
n_samples = raw_beats.shape[1]


def unit_rows(a):
    a = a.reshape(len(a), -1)
    a = a - a.mean(axis=1, keepdims=True)
    return a / np.linalg.norm(a, axis=1, keepdims=True)


def crop(shifts):
    return np.stack([raw_beats[i, SNAP + s:n_samples - SNAP + s] for i, s in enumerate(shifts)])


shifts = np.zeros(len(raw_beats), dtype=int)
for _ in range(5):  # align to the mean beat, recompute the mean, repeat
    ref = unit_rows(crop(shifts).mean(axis=0, keepdims=True))[0]
    corr = np.stack([unit_rows(crop(np.full(len(raw_beats), s))) @ ref for s in range(-SNAP, SNAP + 1)])
    shifts = corr.argmax(axis=0) - SNAP
beat_arr = crop(shifts)  # (n_patients, 60, 12), QRS at sample R_INDEX
R_INDEX = int(0.25 * sampling_rate) - SNAP

# centre + L2-normalise -> Euclidean distance between rows ranks pairs like Pearson / cosine
V = unit_rows(beat_arr)
print(f"{len(first_qc)} patients, {len(V)} after the quality filter")

sample = np.random.default_rng(0).choice(len(V), 5000, replace=False)
silhouette = pd.Series({
    k: silhouette_score(V[sample], KMeans(k, n_init=10, random_state=0).fit_predict(V)[sample], metric="cosine")
    for k in range(2, 9)
}, name="silhouette")
silhouette.round(3).to_frame().T

In [ ]:
# k = 4: silhouette is low for every k (beat shapes vary continuously, no sharp groups),
# 4 keeps the result comparable with the first attempt and separates a clearly abnormal group
N_CLUSTERS = 4
labels = KMeans(N_CLUSTERS, n_init=10, random_state=0).fit_predict(V)

# number clusters 1..k from most to least "normal" so the order is stable and readable
clusters = Y.loc[beat_ids, ["patient_id", "age", "sex", "diagnostic_superclass", "sinus_rhythm"]].copy()
clusters["raw_label"] = labels
norm_share = clusters.groupby("raw_label").diagnostic_superclass.apply(lambda s: s.apply(lambda x: "NORM" in x).mean())
relabel = {old: new + 1 for new, old in enumerate(norm_share.sort_values(ascending=False).index)}
clusters["cluster"] = clusters.raw_label.map(relabel)
clusters = clusters.drop(columns="raw_label")
clusters["similarity_to_centre"] = [  # Pearson between patient's beat and cluster mean beat
    V[i] @ (c := V[clusters.cluster.values == clusters.cluster.values[i]].mean(0)) / np.linalg.norm(c) for i in range(len(V))
]
clusters.to_csv("median_beat_clusters.csv")

superclasses = ["NORM", "MI", "STTC", "CD", "HYP"]
profile = clusters.groupby("cluster").agg(
    patients=("patient_id", "size"),
    median_age=("age", "median"),
    female_pct=("sex", lambda s: s.mean() * 100),
    sinus_pct=("sinus_rhythm", lambda s: s.mean() * 100),
    median_similarity=("similarity_to_centre", "median"),
)
for sc in superclasses:  # an ECG can have several superclasses, so these don't sum to 100
    profile[f"{sc}_pct"] = clusters.groupby("cluster").diagnostic_superclass.apply(lambda s: s.apply(lambda x: sc in x).mean() * 100)
profile.round(1)

In [ ]:
# plots: how many groups, what the groups look like, and how patients are distributed across them
CLUSTER_COLORS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100"]  # categorical slots 1-4, fixed order
cluster_ids = list(range(1, N_CLUSTERS + 1))

# 1) silhouette per k + 2D map of all patients
fig, (ax_k, ax_map) = plt.subplots(1, 2, figsize=(14, 5), width_ratios=[1, 2], constrained_layout=True)
ax_k.plot(silhouette.index, silhouette.values, color=INK_2, linewidth=2, marker="o", markersize=8)
ax_k.plot(N_CLUSTERS, silhouette[N_CLUSTERS], marker="o", markersize=10, color=CLUSTER_COLORS[0])
ax_k.annotate(f"chosen k = {N_CLUSTERS}", (N_CLUSTERS, silhouette[N_CLUSTERS]), xytext=(8, 8), textcoords="offset points")
ax_k.set_xlabel("number of clusters (k)")
ax_k.set_title("Silhouette (cosine): low for every k", loc="left")

pc = PCA(2, random_state=0).fit(V)
xy = pc.transform(V)
for cl, color in zip(cluster_ids, CLUSTER_COLORS):
    m = clusters.cluster.values == cl
    ax_map.scatter(xy[m, 0], xy[m, 1], s=4, alpha=0.35, color=color, linewidths=0, label=f"cluster {cl}  (n={m.sum():,})")
ax_map.set_xlabel(f"PC1 ({pc.explained_variance_ratio_[0]:.0%} of variance)")
ax_map.set_ylabel(f"PC2 ({pc.explained_variance_ratio_[1]:.0%})")
ax_map.set_title("All patients, by median-beat shape (PCA)", loc="left")
ax_map.legend(frameon=False, markerscale=4, loc="upper right")
ax_map.grid(axis="x", visible=True)
fig.savefig("median_beat_clusters_map.png", dpi=120)
plt.show()

# 2) average beat per cluster, 12 leads
lead_names = ["I", "II", "III", "aVR", "aVL", "aVF", "V1", "V2", "V3", "V4", "V5", "V6"]
t_ms = (np.arange(beat_arr.shape[1]) - R_INDEX) * 1000 / sampling_rate
fig, axes = plt.subplots(2, 6, figsize=(16, 6), sharex=True, constrained_layout=True)
for lead, ax in enumerate(axes.ravel()):
    for cl, color in zip(cluster_ids, CLUSTER_COLORS):
        ax.plot(t_ms, beat_arr[clusters.cluster.values == cl, :, lead].mean(0), color=color, linewidth=2, label=f"cluster {cl}")
    ax.set_title(lead_names[lead], loc="left")
    ax.axvline(0, color=GRID, linewidth=1)
for ax in axes[1]:
    ax.set_xlabel("ms from QRS peak")
axes[0, 0].set_ylabel("mV"); axes[1, 0].set_ylabel("mV")
fig.legend(*axes[0, 0].get_legend_handles_labels(), frameon=False, loc="outside upper right", ncols=N_CLUSTERS)
fig.suptitle("Average median beat per cluster", x=0.01, ha="left", fontsize=12)
fig.savefig("median_beat_clusters_beats.png", dpi=120)
plt.show()

# 3) distribution of patients in each cluster: diagnosis, age, sex, similarity to cluster centre
fig, axes = plt.subplots(1, 4, figsize=(17, 4.5), width_ratios=[2, 1.3, 1, 1.3], constrained_layout=True)
ax = axes[0]
width = 0.8 / N_CLUSTERS
x = np.arange(len(superclasses))
for j, (cl, color) in enumerate(zip(cluster_ids, CLUSTER_COLORS)):
    ax.bar(x + (j - (N_CLUSTERS - 1) / 2) * width, profile.loc[cl, [f"{s}_pct" for s in superclasses]],
           width=width * 0.9, color=color, label=f"cluster {cl}")
ax.set_xticks(x, superclasses)
ax.set_ylabel("% of patients in cluster")
ax.set_title("Diagnostic superclass", loc="left")
ax.legend(frameon=False, ncols=2)

for ax, col, title in [(axes[1], "age", "Age"), (axes[3], "similarity_to_centre", "Pearson similarity to cluster centre")]:
    data = [clusters.loc[clusters.cluster == cl, col].dropna() for cl in cluster_ids]
    parts = ax.violinplot(data, positions=cluster_ids, showmedians=True, showextrema=False)
    for body, color in zip(parts["bodies"], CLUSTER_COLORS):
        body.set_facecolor(color); body.set_edgecolor(color); body.set_alpha(0.5)
    parts["cmedians"].set_color(INK)
    ax.set_xticks(cluster_ids, [f"cluster {cl}" for cl in cluster_ids])
    ax.set_title(title, loc="left")

ax = axes[2]
ax.bar(cluster_ids, profile.female_pct, color=CLUSTER_COLORS, width=0.6)
for cl in cluster_ids:
    ax.text(cl, profile.female_pct[cl] + 1, f"{profile.female_pct[cl]:.0f}%", ha="center", color=INK_2)
ax.axhline(clusters.sex.mean() * 100, color=MUTED, linestyle="--", linewidth=1)
ax.set_xticks(cluster_ids, [f"cluster {cl}" for cl in cluster_ids])
ax.set_ylim(0, 70)
ax.set_title("% female (dashed = all patients)", loc="left")
fig.suptitle("How patients are distributed across the median-beat clusters", x=0.01, ha="left", fontsize=12)
fig.savefig("median_beat_clusters_distribution.png", dpi=120)
plt.show()

#### Median-beat clustering after cleaning: results

18,227 patients (first ECG, after the quality filter), k-means with k = 4 on centred + normalised beats (ranks pairs like Pearson / cosine similarity).

| Cluster | Patients | Median age | Female | Main diagnoses | Median similarity to cluster centre |
|---|---|---|---|---|---|
| 1 | 6,065 | 52 | 55% | 67% NORM | 0.86 |
| 2 | 5,270 | 63 | 47% | 50% NORM, mixed | 0.86 |
| 3 | 4,498 | 65 | 47% | mixed: 38% NORM, 30% CD, 29% MI | 0.83 |
| 4 | 2,394 | 66 | 43% | 56% MI, 46% CD, 34% STTC, 23% HYP | 0.81 |

**Compared with the first attempt (before cleaning)**
- The old cluster 4 (1,670 patients, 66% CD, median similarity to its centre only **0.6**) was largely produced by misalignment. The most abnormal group is now larger, mixes MI, CD and hypertrophy, and is much tighter (**0.81**).
- The overall pattern holds: from cluster 1 to 4, NORM falls, MI / CD rise, age rises and % female falls. These are real differences, not effects of the cleaning.
- Separation scores stay low (0.15–0.23 for k = 2–8, vs 0.13–0.21 before). Beat shape varies continuously across patients; treat the clusters as regions of that range, not distinct types.

**Outputs**: `cleaning_test.png`, `median_beat_clusters.csv`, `median_beat_clusters_map.png`, `median_beat_clusters_beats.png`, `median_beat_clusters_distribution.png`.

In [ ]:
#

In [ ]:
# calculate similarity between two signals
# use the metrics to train a classification model to divide patients into groups and find if there's distinct differences

## Watch ECG recommendation model

Input: user info (age, sex, height / weight, diagnosed label) + **two single-lead watch ECG recordings taken at different times**.
Watch ECGs are single-lead (≈ lead I), so every model here is trained on **lead I only**, at 100 Hz; watch recordings of any length / sampling rate are resampled to match.
Train on `strat_fold` 1–8, test on 9–10 (no patient in both).

Built one recommendation type at a time:
1. **ECG group + profile** (this section): which median-beat group each recording falls in, what is typical for that group, and whether the beat shape changed between the two recordings
2. Health / lifestyle actions (next)
3. Diagnosis-risk scores (next)

> Not a medical device: the output describes similarity to PTB-XL patients, it is not a diagnosis.

In [ ]:
# Step 1 - ECG group + profile: lead-I median beats with the cleaned process, groups fitted on training folds
from sklearn.metrics import adjusted_rand_score

WATCH_LEAD = 0  # lead I is the 12-lead equivalent of a single-lead watch ECG
TRAIN_FOLDS, TEST_FOLDS = list(range(1, 9)), [9, 10]


from watch_pipeline import WatchModel, align, plain, to_unit, watch_beat  # shared with the API

w_beats = {eid: watch_beat(X[Y.index.get_loc(eid), :, WATCH_LEAD], sampling_rate) for eid in qc_ids}
w_usable = pd.Series({eid: t is not None and t["usable"] for eid, t in w_beats.items()})
w_usable &= Y.loc[w_usable.index, "pacemaker"].isna() & Y.loc[w_usable.index, "electrodes_problems"].isna()

first_ecg = Y.index[Y.visit_number == 1]
w_train_ids = [e for e in first_ecg[Y.loc[first_ecg, "strat_fold"].isin(TRAIN_FOLDS)] if w_usable[e]]
w_test_ids = [e for e in first_ecg[Y.loc[first_ecg, "strat_fold"].isin(TEST_FOLDS)] if w_usable[e]]
print(f"lead I usable: {w_usable.loc[first_ecg].sum()} of {len(first_ecg)} first ECGs "
      f"-> train {len(w_train_ids)}, test {len(w_test_ids)} patients")

B_train = np.stack([w_beats[e]["beat"][:, 0] for e in w_train_ids])
w_ref = to_unit(B_train[:, SNAP:n_samples - SNAP].mean(axis=0, keepdims=True))[0]
for _ in range(5):  # common reference beat: align, average, repeat
    w_ref = to_unit(align(B_train, w_ref).mean(axis=0, keepdims=True))[0]
W_train = to_unit(align(B_train, w_ref))

w_sample = np.random.default_rng(0).choice(len(W_train), 5000, replace=False)
w_silhouette = pd.Series({
    k: silhouette_score(W_train[w_sample], KMeans(k, n_init=10, random_state=0).fit_predict(W_train)[w_sample], metric="cosine")
    for k in range(2, 9)
}, name="silhouette (lead I)")
w_silhouette.round(3).to_frame().T

In [ ]:
# fit k = 4 groups (same as the 12-lead clustering), number them from most to least "normal", profile on train
W_K = 4
w_km = KMeans(W_K, n_init=10, random_state=0).fit(W_train)
has = lambda ids, sc: Y.loc[ids, "diagnostic_superclass"].apply(lambda x: sc in x)
w_norm_share = pd.Series(has(w_train_ids, "NORM").values).groupby(w_km.labels_).mean()
w_relabel = {old: new + 1 for new, old in enumerate(w_norm_share.sort_values(ascending=False).index)}


# trained values go into one bundle as they are fitted; WatchModel (watch_pipeline.py) reads them at call time,
# so the notebook runs exactly the code the API runs. Saved as watch_model.joblib at the end.
bundle = {"version": "2026-10-06", "sampling_rate": sampling_rate, "watch_lead": WATCH_LEAD,
          "step1": {"reference_beat": w_ref, "kmeans": w_km, "relabel": w_relabel}}
wm = WatchModel(bundle)
assign_group = wm.assign_group


def group_profile(ids, groups):
    m = Y.loc[ids].assign(group=groups)
    prof = m.groupby("group").agg(patients=("patient_id", "size"), median_age=("age", "median"),
                                  female_pct=("sex", lambda s: s.mean() * 100))
    for sc in superclasses:
        prof[f"{sc}_pct"] = m.diagnostic_superclass.apply(lambda x: sc in x).groupby(m.group).mean() * 100
    return prof


w_train_groups = np.array([w_relabel[l] for l in w_km.labels_])
w_test_groups = np.array([assign_group(w_beats[e]["beat"][:, 0])[0] for e in w_test_ids])
w_profile = group_profile(w_train_ids, w_train_groups)
w_profile_test = group_profile(w_test_ids, w_test_groups)

# how much a person's own lead-I beat normally varies between recordings (training 2-ECG patients)
own_pairs = pairs_df[(pairs_df.pair == "same patient")
                     & Y.loc[pairs_df.ecg_1, "strat_fold"].isin(TRAIN_FOLDS).values
                     & pairs_df.ecg_1.map(w_usable).values & pairs_df.ecg_2.map(w_usable).values]
own_similarity = [shift_pearson(w_beats[a]["beat"], w_beats[b]["beat"]) for a, b in zip(own_pairs.ecg_1, own_pairs.ecg_2)]
CHANGE_THRESHOLD = np.percentile(own_similarity, 5)  # below this = more change than 95% of repeat recordings
bundle["step1"].update(profile=w_profile, change_threshold=CHANGE_THRESHOLD)
bundle["peers"] = Y.loc[w_train_ids, ["age", "sex", "diagnostic_superclass"]].assign(group=w_train_groups)
print(f"change threshold (5th percentile of {len(own_similarity)} training repeat pairs): r = {CHANGE_THRESHOLD:.3f}")
pd.concat({"train": w_profile, "test": w_profile_test}).round(1)

In [ ]:
# recommendation function, step 1: watch_pipeline.WatchModel.recommend (shared with the API)
# recommend(user, rec_1, rec_2) -> group + profile, beat-shape change, peers, plain-text recommendations
# user: {"age", "sex": "female"/"male", "height_cm", "weight_kg", "diagnosed_label": NORM/MI/STTC/CD/HYP or None}
# rec_1, rec_2: {"signal": 1-D array (single lead, mV), "fs": sampling rate in Hz}; rec_1 is the older recording
recommend = wm.recommend

In [ ]:
# evaluate step 1 on the test folds (9-10)
# a) are a person's two recordings put in the same group?  b) does the change flag fire on own vs mismatched pairs?
# c) do the lead-I groups agree with the 12-lead groups?
test_pairs = pairs_df[Y.loc[pairs_df.ecg_1, "strat_fold"].isin(TEST_FOLDS).values
                      & pairs_df.ecg_1.map(w_usable).values & pairs_df.ecg_2.map(w_usable).values].copy()
test_pairs["similarity"] = [shift_pearson(w_beats[a]["beat"], w_beats[b]["beat"]) for a, b in zip(test_pairs.ecg_1, test_pairs.ecg_2)]
test_pairs["flagged"] = test_pairs.similarity < CHANGE_THRESHOLD
test_pairs["same_group"] = [assign_group(w_beats[a]["beat"][:, 0])[0] == assign_group(w_beats[b]["beat"][:, 0])[0]
                            for a, b in zip(test_pairs.ecg_1, test_pairs.ecg_2)]

group_share = pd.Series(w_test_groups).value_counts(normalize=True)
both12 = clusters.index.intersection(w_test_ids)
w_eval = pd.DataFrame({
    "own pairs (same patient)": {
        "pairs": (test_pairs.pair == "same patient").sum(),
        "both recordings in same group %": test_pairs[test_pairs.pair == "same patient"].same_group.mean() * 100,
        "change flagged %": test_pairs[test_pairs.pair == "same patient"].flagged.mean() * 100,
    },
    "mismatched pairs": {
        "pairs": (test_pairs.pair == "different patients").sum(),
        "both recordings in same group %": test_pairs[test_pairs.pair == "different patients"].same_group.mean() * 100,
        "change flagged %": test_pairs[test_pairs.pair == "different patients"].flagged.mean() * 100,
    },
}).T
w_eval["same group by chance %"] = (group_share ** 2).sum() * 100
print(f"change-detection AUC (test): {roc_auc_score(test_pairs.pair == 'same patient', test_pairs.similarity):.3f}")
print(f"lead-I vs 12-lead groups (test patients in both, n={len(both12)}): adjusted Rand index "
      f"{adjusted_rand_score(clusters.loc[both12, 'cluster'], pd.Series(w_test_groups, index=w_test_ids).loc[both12]):.3f}")
w_eval.round(1)

In [ ]:
# plot step 1: what each lead-I group looks like, its diagnosis profile (train vs test), and the change threshold
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), width_ratios=[1, 1.5, 1.2], constrained_layout=True)
ax = axes[0]
t_w = (np.arange(W_train.shape[1]) - R_INDEX) * 1000 / sampling_rate
for g, color in zip(range(1, W_K + 1), CLUSTER_COLORS):
    ax.plot(t_w, align(B_train[w_train_groups == g], w_ref).mean(axis=0), color=color, linewidth=2,
            label=f"group {g} (n={(w_train_groups == g).sum():,})")
ax.axvline(0, color=GRID, linewidth=1)
ax.set_xlabel("ms from QRS peak"); ax.set_ylabel("mV")
ax.set_title("Average lead-I beat per group (train)", loc="left")
ax.legend(frameon=False, fontsize=8)

ax = axes[1]
width = 0.8 / W_K
x = np.arange(len(superclasses))
for j, (g, color) in enumerate(zip(range(1, W_K + 1), CLUSTER_COLORS)):
    pos = x + (j - (W_K - 1) / 2) * width
    ax.bar(pos, w_profile.loc[g, [f"{s}_pct" for s in superclasses]], width=width * 0.9, color=color, label=f"group {g}")
    ax.scatter(pos, w_profile_test.loc[g, [f"{s}_pct" for s in superclasses]], color=INK, s=14, zorder=3,
               label="test folds" if j == 0 else None)
ax.set_xticks(x, superclasses)
ax.set_ylabel("% of patients in group")
ax.set_title("Diagnosis profile per group: bars = train, dots = test", loc="left")
ax.legend(frameon=False, ncols=3, fontsize=8)

ax = axes[2]
bins = np.linspace(-0.2, 1, 49)
for pair, color in PAIR_COLORS.items():
    ax.hist(test_pairs.loc[test_pairs.pair == pair, "similarity"].clip(-0.2, 1), bins=bins, density=True,
            histtype="step", linewidth=2, color=color, label=pair)
ax.axvline(CHANGE_THRESHOLD, color=INK_2, linestyle="--", linewidth=1)
ax.text(CHANGE_THRESHOLD - 0.02, ax.get_ylim()[1] * 0.92, f"change threshold\nr = {CHANGE_THRESHOLD:.2f}", ha="right", color=INK_2)
ax.set_yticks([])
ax.set_xlabel("shift-tolerant Pearson r between the two lead-I beats")
ax.set_title("Change between two recordings (test)", loc="left")
ax.legend(frameon=False, loc="upper left")
fig.suptitle("Step 1 - lead-I (watch) ECG groups", x=0.01, ha="left", fontsize=12)
fig.savefig("watch_step1_groups.png", dpi=120)
plt.show()

In [ ]:
# demo: simulate a watch - lead I at 500 Hz from the high-resolution records of a test patient with two ECGs
demo = test_pairs[(test_pairs.pair == "same patient")].iloc[0]
watch_rec = lambda eid: {"signal": wfdb.rdsamp(os.path.join(ecg_db_path, Y.filename_hr[eid]), channels=[WATCH_LEAD])[0][:, 0], "fs": 500}
p1 = Y.loc[demo.ecg_1]
demo_user = {
    "age": p1.age, "sex": "female" if p1.sex == 1 else "male",
    "height_cm": p1.height if p1.height == p1.height else None, "weight_kg": p1.weight if p1.weight == p1.weight else None,
    "diagnosed_label": next(iter(p1.diagnostic_superclass), None),  # label from their earlier ECG
}
demo_user = plain(demo_user)
print("user:", demo_user)
result = recommend(demo_user, watch_rec(demo.ecg_1), watch_rec(demo.ecg_2))
for line in result["recommendations"]:
    print("-", line)
result

#### Step 1 - ECG group + profile: results

**Lead I (watch) groups**: 18,203 of 18,869 patients have a usable lead-I beat; groups fitted on 14,496 training patients (folds 1–8), checked on 3,707 test patients (folds 9–10).

| Group | Train patients | Median age | Female | NORM | MI | STTC | CD | HYP | Lead-I beat shape |
|---|---|---|---|---|---|---|---|---|---|
| 1 | 3,609 | 51 | 48% | **75%** | 11% | 6% | 13% | 4% | tall R, normal T |
| 2 | 2,043 | 51 | 47% | 58% | 17% | 9% | **33%** | 7% | deep S after the QRS |
| 3 | 6,453 | 63 | 52% | 45% | 24% | 26% | 15% | 11% | smaller, flatter T |
| 4 | 2,391 | 68 | 43% | **3%** | 48% | **51%** | 45% | **31%** | inverted T-wave |

**Evaluation on the test folds**
- **Groups generalise**: the diagnosis profile of each group in the test folds matches training within a few points (dots vs bars in the plot).
- **Groups are sharper than with 12 leads** (silhouette 0.28 at k = 4 vs 0.17), and group 4 is almost entirely abnormal (3% NORM).
- **They are different groups from the 12-lead clusters** (adjusted Rand index 0.06): one lead sees the T-wave and S-wave in lead I, not the whole 12-lead shape.
- **Stability**: a person's two recordings land in the same group **74%** of the time (29% by chance). The other 26% sit near a boundary, so the recommendation says so and uses the latest recording.
- **Change detection** (shift-tolerant Pearson between the two beats, threshold r = 0.843 = 5th percentile of training repeat pairs): AUC **0.897**; flags **7.8%** of a person's own repeat pairs (false alarms) and **57%** of pairs from two different people.

**What `recommend(user, rec_1, rec_2)` returns**: per-recording quality / group / heart rate, the group's diagnosis profile, the change check, peers (same group, same sex, age ±5) and how common the user's diagnosed label is among them vs overall, BMI (reported only: height / weight are too sparse in PTB-XL to train on), and plain-text recommendations.

**Limits**
- PTB-XL lead I stands in for a watch: real watch ECGs are noisier and taken with a finger on the crown, so thresholds should be re-checked on real watch data.
- Recommendations describe similarity to PTB-XL patients; they are not a diagnosis.

**Output**: `watch_step1_groups.png`

### Step 2 - Health / lifestyle actions (DRAFT rules for review)

PTB-XL has no labels for "the right action", so step 2 is a **rule set** on top of step 1. Thresholds are set from PTB-XL where a label exists to check against (atrial fibrillation, slow / fast rhythm, normal HRV by age), otherwise from common clinical cut-offs. Every threshold lives in `RULES` so it can be changed in one place.

**Inputs per watch recording** (lead I, RR intervals at the watch's own sampling rate, no resampling): heart rate, RMSSD, SDNN, irregularity; plus the step-1 result (group, beat-shape change) and user info.

**Action levels** (the highest-priority action is shown first): 1 seek care promptly · 2 see a clinician · 3 re-record · 4 lifestyle · 5 routine

| # | Rule | Condition | Level | Action |
|---|---|---|---|---|
| R1 | Quality | a recording is unusable | 3 | re-record: sit still 1 min, arm resting, finger on the electrode throughout |
| R2 | Very slow / fast | resting HR < 40 or > 130 bpm | 1 | seek medical advice promptly; urgently with dizziness, fainting, chest pain, breathlessness |
| R3 | Fast | HR 100–130 | 3 | rest 5 min and re-record; if still > 100 at rest, see a clinician |
| R4 | Slow | HR 40–50 | 4 | common if fit; with dizziness / tiredness, see a clinician |
| R5 | Irregular rhythm | irregularity > AF threshold (set at 95% specificity on training data) in **both** recordings | 2 | possible atrial fibrillation: see a clinician, share the recordings |
| R6 | Irregular once | irregular in the latest recording only | 3 | re-record at rest; if irregular again, see a clinician |
| R7 | Beat-shape change | step-1 change flag | 2 | see a clinician if it persists on re-recording |
| R8 | Abnormal-like group | step-1 group 4, no known diagnosis | 2 | ask a clinician about a 12-lead ECG |
| R9 | Known diagnosis | diagnosed label other than NORM | 4 | follow your clinician's plan, bring recordings to check-ups |
| R10 | HR differs | HR differs > 20 bpm between recordings | 3 | record at rest, same time of day, so recordings compare |
| R11 | Low HRV | regular rhythm, HR 50–100, RMSSD below the 10th percentile for age (normal sinus-rhythm PTB-XL patients) | 4 | recovery day: sleep, hydration, lower stress, light activity |
| R12 | Exercise | any level ≤ 2 → avoid strenuous exercise until reviewed; group 3/4 or age ≥ 65 → moderate, build up gradually; low HRV → light today; else → ≥ 150 min moderate / week | 4–5 | |
| R13 | BMI | BMI ≥ 30 or < 18.5 | 4 | weight guidance; discuss with a clinician / dietitian |
| R14 | Routine | nothing above level 4 | 5 | record weekly, at rest, same time of day |

In [ ]:
# Step 2 - rhythm features from a single-lead watch recording, calibrated on PTB-XL lead I
from watch_pipeline import rhythm_features, watch_rr  # shared with the API

rf = {}
for eid in w_usable.index[w_usable]:
    f = rhythm_features(watch_rr(X[Y.index.get_loc(eid), :, WATCH_LEAD], sampling_rate))
    if f is not None:
        rf[eid] = f
w_rhythm = pd.DataFrame.from_dict(rf, orient="index")
codes = Y.loc[w_rhythm.index, "scp_codes"]
w_rhythm["af"] = codes.apply(lambda c: bool({"AFIB", "AFLT"} & set(c)))  # atrial fibrillation / flutter
w_rhythm["age"] = Y.loc[w_rhythm.index, "age"]
rh_train = w_rhythm.loc[w_rhythm.index.intersection(w_train_ids)]
rh_test = w_rhythm.loc[w_rhythm.index.intersection(w_test_ids)]

# AF rule: pick the better irregularity feature, threshold at 95% specificity on training non-AF ECGs
af_auc = {f: roc_auc_score(rh_train.af, rh_train[f]) for f in ["nrmssd", "irregularity"]}
AF_FEATURE = max(af_auc, key=af_auc.get)
AF_THRESHOLD = rh_train.loc[~rh_train.af, AF_FEATURE].quantile(0.95)

# low-HRV rule: 10th percentile of RMSSD by age band, normal sinus-rhythm training patients with regular rhythm
AGE_BANDS = [0, 30, 40, 50, 60, 70, 120]
is_ref = (~rh_train.af & codes.loc[rh_train.index].apply(lambda c: "SR" in c)
          & Y.loc[rh_train.index, "diagnostic_superclass"].apply(lambda x: x == ["NORM"])
          & (rh_train[AF_FEATURE] <= AF_THRESHOLD))
hrv_ref = rh_train[is_ref]
hrv_norms = hrv_ref.groupby(pd.cut(hrv_ref.age, AGE_BANDS, right=False), observed=False).rmssd_ms.describe(percentiles=[0.1, 0.5])[["count", "10%", "50%"]]


bundle["step2"] = {"af_feature": AF_FEATURE, "af_threshold": AF_THRESHOLD, "hrv_norms": hrv_norms,
                  "age_bands": AGE_BANDS, "rmssd_low_overall": hrv_ref.rmssd_ms.quantile(0.10)}
low_rmssd_cutoff = wm.low_rmssd_cutoff


print(f"AF feature AUC (train): {af_auc} -> using {AF_FEATURE}, threshold {AF_THRESHOLD:.3f}")
print(f"AF on test: sensitivity {(rh_test.loc[rh_test.af, AF_FEATURE] > AF_THRESHOLD).mean():.1%}, "
      f"specificity {(rh_test.loc[~rh_test.af, AF_FEATURE] <= AF_THRESHOLD).mean():.1%}, "
      f"AUC {roc_auc_score(rh_test.af, rh_test[AF_FEATURE]):.3f} (n AF = {rh_test.af.sum()})")
hrv_norms.round(1)

In [ ]:
# Step 2 - rules (DRAFT): every threshold in one place
RULES = {
    "hr_urgent_low": 40, "hr_urgent_high": 130,  # bpm, R2
    "hr_low": 50, "hr_high": 100,                # bpm, R3 / R4 / R11
    "hr_change": 20,                             # bpm between recordings, R10
    "bmi_low": 18.5, "bmi_high": 30,             # R13
    "senior_age": 65,                            # R12
}
from watch_pipeline import LEVELS

# the 14 rules (R1-R14) are watch_pipeline.WatchModel.health_actions (shared with the API); thresholds come from RULES
bundle["step2"]["rules"] = RULES
health_actions = wm.health_actions

In [ ]:
# evaluate step 2 on the test folds: rule vs label agreement, and which actions fire for normal vs abnormal patients
rhythm_labels = {"SBRAD": "sinus bradycardia", "STACH": "sinus tachycardia"}
label_check = pd.DataFrame({
    "sinus bradycardia (SBRAD)": {"ECGs": codes.loc[rh_test.index].apply(lambda c: "SBRAD" in c).sum(),
                                  "rule fires %": (rh_test.heart_rate[codes.loc[rh_test.index].apply(lambda c: "SBRAD" in c)] < RULES["hr_low"]).mean() * 100},
    "sinus tachycardia (STACH)": {"ECGs": codes.loc[rh_test.index].apply(lambda c: "STACH" in c).sum(),
                                  "rule fires %": (rh_test.heart_rate[codes.loc[rh_test.index].apply(lambda c: "STACH" in c)] > RULES["hr_high"]).mean() * 100},
    "atrial fibrillation / flutter": {"ECGs": rh_test.af.sum(),
                                      "rule fires %": (rh_test.loc[rh_test.af, AF_FEATURE] > AF_THRESHOLD).mean() * 100},
    "other ECGs (false alarm rate for AF)": {"ECGs": (~rh_test.af).sum(),
                                             "rule fires %": (rh_test.loc[~rh_test.af, AF_FEATURE] > AF_THRESHOLD).mean() * 100},
}).T

# full pipeline on every test patient with two ECGs; diagnosed label left out so it can't leak the answer
pipeline_rows = []
for r in test_pairs[test_pairs.pair == "same patient"].itertuples():
    p_ = Y.loc[r.ecg_1]
    user = {"age": p_.age, "sex": "female" if p_.sex == 1 else "male", "diagnosed_label": None,
            "height_cm": p_.height if p_.height == p_.height else None, "weight_kg": p_.weight if p_.weight == p_.weight else None}
    res = health_actions(user, {"signal": X[Y.index.get_loc(r.ecg_1), :, WATCH_LEAD], "fs": sampling_rate},
                         {"signal": X[Y.index.get_loc(r.ecg_2), :, WATCH_LEAD], "fs": sampling_rate})
    latest_labels = Y.loc[r.ecg_2, "diagnostic_superclass"]
    pipeline_rows.append({"patient_id": r.patient_id, "top_level": res["top_action"]["level"],
                          "rules": [a["rule"] for a in res["actions"]],
                          "latest ECG": "normal (NORM only)" if latest_labels == ["NORM"] else "abnormal"})
pipeline_eval = pd.DataFrame(pipeline_rows)
top_level_share = (pd.crosstab(pipeline_eval["latest ECG"], pipeline_eval.top_level, normalize="index") * 100) \
    .rename(columns=lambda l: f"{l} {LEVELS[l]}")
rule_rate = pipeline_eval.explode("rules").groupby("latest ECG").rules.value_counts().unstack(fill_value=0) \
    .div(pipeline_eval["latest ECG"].value_counts(), axis=0).mul(100)
print(f"{len(pipeline_eval)} test patients with two ECGs")
display(label_check.round(1), top_level_share.round(1), rule_rate.round(1))

In [ ]:
# plot step 2: AF rule, HRV norms by age, top action for normal vs abnormal test patients
LEVEL_COLORS = {1: "#d03b3b", 2: "#ec835a", 3: "#fab219", 4: "#86b6ef", 5: "#0ca30c"}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), width_ratios=[1.2, 1, 1.4], constrained_layout=True)

ax = axes[0]
hi = rh_test[AF_FEATURE].quantile(0.99)
bins = np.linspace(0, hi, 40)
for is_af, color, name in [(False, "#2a78d6", "no AF"), (True, "#eb6834", "AF / flutter")]:
    ax.hist(rh_test.loc[rh_test.af == is_af, AF_FEATURE].clip(0, hi), bins=bins, density=True, histtype="step",
            linewidth=2, color=color, label=f"{name} (n={(rh_test.af == is_af).sum():,})")
ax.axvline(AF_THRESHOLD, color=INK_2, linestyle="--", linewidth=1)
ax.text(AF_THRESHOLD * 1.05, ax.get_ylim()[1] * 0.9, f"threshold {AF_THRESHOLD:.2f}\n(95% specificity)", color=INK_2)
ax.set_yticks([])
ax.set_xlabel(f"{AF_FEATURE} (RR irregularity)")
ax.set_title("Irregular-rhythm rule (test)", loc="left")
ax.legend(frameon=False, loc="upper right")

ax = axes[1]
band_names = [f"{int(b.left)}-{int(b.right) - 1}" if b.right < 120 else f"{int(b.left)}+" for b in hrv_norms.index]
ax.plot(band_names, hrv_norms["50%"], color="#2a78d6", linewidth=2, marker="o", markersize=8, label="median")
ax.plot(band_names, hrv_norms["10%"], color="#eb6834", linewidth=2, marker="o", markersize=8, label="10th percentile = low-HRV cut-off")
ax.set_xlabel("age band"); ax.set_ylabel("RMSSD (ms)")
ax.set_title("Normal RMSSD by age (lead I, train)", loc="left")
ax.legend(frameon=False)

ax = axes[2]
left = np.zeros(len(top_level_share))
for level in LEVELS:
    col = f"{level} {LEVELS[level]}"
    vals = top_level_share[col].values if col in top_level_share else np.zeros(len(top_level_share))
    ax.barh(top_level_share.index, vals, left=left, color=LEVEL_COLORS[level], label=col, height=0.55,
            edgecolor=SURFACE, linewidth=2)
    for y, (l, v) in enumerate(zip(left, vals)):
        if v >= 6:
            ax.text(l + v / 2, y, f"{v:.0f}%", ha="center", va="center", color=INK, fontsize=9)
    left += vals
ax.set_xlim(0, 100)
ax.set_xlabel("% of test patients (top-priority action)")
ax.set_title("Top action by latest ECG label (test)", loc="left")
ax.legend(frameon=False, fontsize=8, ncols=3, loc="upper center", bbox_to_anchor=(0.5, -0.18))
ax.grid(axis="y", visible=False); ax.grid(axis="x", visible=True)
fig.suptitle("Step 2 - health / lifestyle action rules (draft)", x=0.01, ha="left", fontsize=12)
fig.savefig("watch_step2_actions.png", dpi=120)
plt.show()

In [ ]:
# demo: same simulated watch user as step 1 (500 Hz lead I)
step2 = health_actions(demo_user, watch_rec(demo.ecg_1), watch_rec(demo.ecg_2))
for a in step2["actions"]:
    print(f"[{a['level']} {a['category']}] {a['rule']}: {a['action']}  ({a['reason']})")
step2["rhythm"]

#### Step 2 - Health / lifestyle actions: results (draft rules)

**Calibrated from PTB-XL lead I** (train folds 1–8, checked on test folds 9–10)
- **Irregular rhythm (R5 / R6)**: `irregularity` = median |successive RR difference| / median RR (AUC 0.943 train vs 0.919 for normalised RMSSD). Threshold **0.116** = 95% specificity on training ECGs without AF.
  Test: AUC **0.938**, sensitivity **76%** for atrial fibrillation / flutter (205 ECGs), false alarms **5.3%** of other ECGs.
- **Low HRV (R11)**: 10th percentile of RMSSD in normal sinus-rhythm patients, by age band: 19.5 ms (< 30), 13.8 (30s), 11.5 (40s), 10.5 (50s), 9.4 (60s), 8.0 ms (70+).

**Rules vs PTB-XL labels (test)**

| Label | ECGs | Rule fires |
|---|---|---|
| Sinus tachycardia (STACH) → HR > 100 | 135 | 94% |
| Sinus bradycardia (SBRAD) → HR < 50 | 107 | 52% (by design: SBRAD is < 60 bpm; the alert starts at 50 because 50–60 is common and harmless in fit people) |
| AF / flutter → irregular | 205 | 76% |
| Other ECGs → irregular (false alarm) | 3,502 | 5.3% |

**Full pipeline on the 308 test patients with two ECGs** (diagnosed label left out, so it can't leak the answer)

| Top-priority action | Latest ECG normal | Latest ECG abnormal |
|---|---|---|
| 1 seek care promptly | 0% | 3% |
| 2 see a clinician | 3.5% | 40% |
| 3 re-record | 11% | 13% |
| 4 lifestyle | 44% | 35% |
| 5 routine | 42% | 9% |

- Patients with an abnormal 12-lead ECG get a level 1–2 action **12× more often** (43% vs 3.5%); mainly from R8 (group 4: 33% vs 1%), R7 (beat-shape change: 11% vs 3%) and R5 (irregular in both: 6% vs 1%).
- 57% of abnormal patients still get no level 1–2 action: a single lead can't see every 12-lead finding, so the app should never present "routine" as "all clear".
- Low HRV (R11) fires for 8% of normal patients, close to the 10% the cut-off is designed for.

**Open questions for review**
1. HR alert levels (40 / 50 / 100 / 130 bpm): keep, or adjust for your users (e.g. athletes)?
2. Irregular-rhythm threshold: 95% specificity (fewer false alarms, misses 24% of AF) or more sensitive (e.g. 90% specificity)?
3. Should a known diagnosis (R9) also soften R7 / R5, or keep those alerts regardless?
4. Exercise wording (R12): fine as general guidance, or should it defer entirely to the clinician when any diagnosis is known?

**Limits**: PTB-XL ECGs are 10 s at 100 Hz (RR precision 10 ms); a 30 s watch recording at its own sampling rate gives more beats and more precise HRV, so the AF and HRV thresholds should be re-checked on real watch data. Not a medical device.

**Output**: `watch_step2_actions.png`